### Tools

**Tools** = functions the LLM can decide to call to get information or perform actions it can't do on its own (search the web, run code, query a DB, call an API, etc.).

Workflow:
1. You define a tool (a Python function with a clear name + description).
2. You bind it to the model: `model.bind_tools([my_tool])`.
3. The model, when prompted, decides whether to **call the tool** and with what arguments — it returns a `tool_call` in its response.
4. Your code executes the tool and feeds the result back to the model.

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

model_with_tools = model.bind_tools([add])
response = model_with_tools.invoke("What is 12 + 30?")
print(response.tool_calls)
```

Tools are what turn an LLM into an **agent** — a model that can take actions, not just generate text.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Why do parrots talk?")

response.content

'<think>\nOkay, so the user is asking, "Why do parrots talk?" Hmm, I need to explain the reasons behind parrots\' ability to mimic human speech. Let me start by recalling what I know about parrots. They\'re part of the Psittaciformes order, right? Known for their intelligence and vocal mimicry. \n\nFirst, I should mention their vocal anatomy. Parrots have a syrinx, which is their vocal organ, similar to other birds but with some unique structures that allow for a wider range of sounds. That probably helps them imitate human speech. Then there\'s the social aspect. Parrots are social animals, often living in flocks in the wild. Mimicking sounds could be a way to communicate or bond with others. \n\nAlso, in captivity, parrots might learn to talk to interact with their human companions. They might associate certain words with actions or outcomes, like getting food or attention. Positive reinforcement plays a role here, I think. If a parrot says a word and gets a reward, they\'re more lik

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the Weather at a location"""
    return f"It's sunnny in {location}"

In [4]:
model_with_tool = model.bind_tools([get_weather])

In [9]:
response = model_with_tool.invoke("What's the Weather like in Japan")

print(response.text)

for tool_call in response.tool_calls:
    print(f"Tools : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")


Tools : get_weather
Args : {'location': 'Japan'}


### Tool Execution loop

In [8]:
## Step1: Model generates tool class
messages = [{'role': 'user', 'content': "What's the Weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

## Step2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

## Step 3
final_response = model_with_tool.invoke(messages)

print(final_response.text)


The weather in Boston is sunny.


In [10]:
messages

[{'role': 'user', 'content': "What's the Weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'kyrf5ty9f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 153, 'total_tokens': 239, 'completion_time': 0.146768236, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006904172, 'prompt_tokens_details': None, 'queue_time': 0.226258325, 'total_time': 0.153672408}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l